# 01 - Download: all NYC Citi Bike data (2013 - now)

**Who worked on this:** Mohammed AbuQamar

**What this notebook does:** download every NYC trip archive, unpack it and assemble everything into parquet files with one common set of column names. All the logic lives in `src/download.py`, so the retraining pipeline can run the same code.

**What is in the bucket** (checked on the [Citi Bike S3 bucket](https://s3.amazonaws.com/tripdata/index.html)):
- 2013 - 2023: one zip per **year**; 2024 onwards: one zip per **month**
- about **32 GB of zips** for NYC, roughly 352 million trips
- `JC-...` files are Jersey City, a separate system: skipped

**Two different schemas.** Up to January 2021 the columns are `tripduration, starttime, usertype, birth year, ...` (and in late 2016 they were even written as `Trip Duration, Start Time, ...`). From February 2021 they are `ride_id, rideable_type, started_at, member_casual, ...`. To assemble one dataset we map every header to one canonical name (`src/cleaning.canonical_name`). **Only the names change here; every value stays raw text.** Cleaning the values is done in 02/03.


- **Resumable:** finished archives are written to `raw/_done.txt`. If it crashes or you stop it, just run it again.
- Parquet instead of CSV: we used csv but it was much slower so after we looked into itn we landed on parquet it is smaller, and lighter give less strees on th ram beacuse Duck Db just straem the data

In [2]:
import sys
sys.path.append(".")
import pandas as pd
import duckdb
from src.download import list_archives, download_all, RAW_DIR

## 1. Which archives are there?
Read from the bucket listing itself, so new months are picked up automatically when this runs again later.

In [3]:
archives = list_archives()
print(len(archives), "NYC archives:", archives[0], "...", archives[-1])
pd.Series([a[:4] for a in archives]).value_counts().sort_index().to_frame("archives per year").T

44 NYC archives: 2013-citibike-tripdata.zip ... 202609-citibike-tripdata.zip


,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
archives per year,1,1,1,1,1,1,1,1,1,1,1,12,12,9


## 2. Download and convert
`YEARS = None` downloads everything. To test quickly first, use e.g. `YEARS = [2016, 2024]`: one legacy and one current year. Running it again later with `None` only downloads what is still missing.

In [4]:
YEARS = None
archives = download_all(years=YEARS)
print("Done:", len(archives), "archives")


44 archives, 0 already done, 44 to go
  converting 201309-citibike-tripdata.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  converting 201311-citibike-tripdata.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  converting 201307-citibike-tripdata.csv


...

  converting 202609-citibike-tripdata_2.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  converting 202609-citibike-tripdata_3.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done: 44 archives


## 3. Check the result
Trips per year, and which canonical columns are filled in which period. The schema change of 2021 should be clearly visible.

In [5]:
con = duckdb.connect()
src = f"read_parquet('{RAW_DIR.as_posix()}/**/*.parquet', filename=true, union_by_name=true)"
per_year = con.execute(f'''
    SELECT left(regexp_extract(replace(filename, '\\', '/'), 'raw/([0-9]+)/', 1), 4) AS year,
           count(*) AS trips,
           round(100 * avg((ride_id IS NOT NULL)::INT), 1)     AS pct_with_ride_id,
           round(100 * avg((birth_year IS NOT NULL)::INT), 1)  AS pct_with_birth_year
    FROM {src} GROUP BY 1 ORDER BY 1''').df()
display(per_year)
print(f"Total: {per_year.trips.sum():,} trips")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,trips,pct_with_ride_id,pct_with_birth_year
0,2013,11229776,0.0,97.9
1,2014,8081216,0.0,97.4
2,2015,9937969,0.0,86.8
3,2016,13845655,0.0,88.5
4,2017,16364657,0.0,90.0
5,2018,35096678,0.0,100.0
6,2019,20551697,0.0,100.0
7,2020,19562314,100.0,0.0
8,2021,27130122,100.0,0.0
9,2022,29838806,100.0,0.0


Total: 351,963,720 trips


`pct_with_ride_id` should be ~0 before 2021 and ~100 after, `pct_with_birth_year` the other way around. 2021 contains both (January is still legacy).